# 02 — Translate Candidates to German

Takes the **full deduplicated candidate list** from `00_prepare_candidates.ipynb`
(`output/unique_candidates_english_zipf.csv`) and translates every candidate word to German
using **Google Translate** via `deep_translator` (free, no API key needed), using the
THINGSplus/THINGS definition as context to reduce ambiguity.

Since `01`'s category logic isn't run here, this notebook derives the same
`primary_category` (renamed `category`) from THINGSplus's `cat53_string` directly, so
notebooks `03`/`04` still have a `category` column to group by.


In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm


In [ ]:
candidates_df = pd.read_csv("output/unique_candidates_english_zipf.csv")
things_meta = pd.read_csv("/home/hivrim8h/projects/things_database/concepts-metadata_things.tsv", sep="\t")

# Extract primary category
candidates_df["category"] = candidates_df["cat53_string"].str.split(";").str[0].str.strip()

# Handle uncategorized words
uncategorized = candidates_df[candidates_df["category"].isna()]
if not uncategorized.empty:
    uncategorized.to_csv("output/uncategorized_words_de.csv", index=False)
    print(f"{len(uncategorized)} uncategorized words saved (excluded from translation)")
selected_df = candidates_df.dropna(subset=["category"]).copy()

# 2. Clean 1-to-1 Mapping of words to definition
definition_map = dict(zip(things_meta["uniqueID"], things_meta["Definition (from WordNet, Google, or Wikipedia)"]))
selected_df["Definition"] = selected_df["uniqueID"].map(definition_map)

print(f"{len(selected_df)} words loaded for translation | "
      f"{selected_df['Definition'].isna().sum()} missing a definition")





98 uncategorized words saved (excluded from translation)
371 words loaded for translation | 0 missing a definition


In [ ]:
# 3. Translate to German
translator = GoogleTranslator(source="en", target="de")
cache = {}

def translate_row(row):
    term = row["Word"]
    definition = row["Definition"]
    
    # Context format or fallback to simple term if definition is missing
    context = f"{term}: {definition}" if pd.notna(definition) else term
    
    if context in cache:
        return cache[context]
    
    try:
        translated = translator.translate(context)
        # Verify the translator returned a colon-separated translation
        if pd.notna(definition) and ":" not in str(translated):
            # If Google Translator dropped the format structure, retry translating the term only
            translated = translator.translate(term)
    except Exception as e:
        # Fallback to word-only translation directly on error
        try:
            translated = translator.translate(term)
        except Exception:
            translated = None
            
    cache[context] = translated
    return translated

tqdm.pandas(desc="Translating")
selected_df["GermanTerm"] = selected_df.progress_apply(translate_row, axis=1)

Translating: 100%|██████████| 371/371 [04:30<00:00,  1.37it/s]


In [4]:
# ===== Retry failures with word-only translation =====
retry_mask = selected_df["GermanTerm"].isna()
print(f"{retry_mask.sum()} translations failed on first pass, retrying word-only")

for idx in selected_df[retry_mask].index:
    term = selected_df.at[idx, "Word"]
    if term in cache:
        result = cache[term]
    else:
        try:
            result = translator.translate(term)
        except Exception as e:
            print(f"Retry translation error: {term} -> {e}")
            result = None
        cache[term] = result
    selected_df.at[idx, "GermanTerm"] = result

remaining = selected_df[selected_df["GermanTerm"].isna()]
if not remaining.empty:
    remaining.to_csv("output/translation_errors_final_selection.tsv", sep="\t", index=False)
    print(f"{len(remaining)} words still failed, saved to output/translation_errors_final_selection.tsv")
else:
    print("All words translated successfully.")


# Handle final failures (if any)
failed_mask = selected_df["GermanTerm"].isna()
if failed_mask.any():
    selected_df[failed_mask].to_csv("output/translation_errors_final_selection.tsv", sep="\t", index=False)
    print(f"{failed_mask.sum()} words still failed, logged to output/translation_errors_final_selection.tsv")


0 translations failed on first pass, retrying word-only
All words translated successfully.


In [ ]:

# 4. Extract Translation & Clean up
selected_df["GermanTerm"] = selected_df["GermanTerm"].fillna("")
split_cols = selected_df["GermanTerm"].str.split(":", n=1, expand=True)

# If translation is 'term: meaning', split it. Otherwise, use the entire string as the translation.
selected_df["GermanTranslation"] = split_cols[0].str.strip()
selected_df["word_de"] = selected_df["GermanTranslation"].str.lower()

# 5. Export clean selection tsv
output_cols = ["word_de", "Word", "GermanTranslation", "category"]
selected_df[output_cols].to_csv("output/english_to_german_selection.tsv", sep="\t", index=False)
print("Saved output/english_to_german_selection.tsv")


Saved output/english_to_german_selection.tsv
